# Pipeline ML — GlicNutri (paciente-dia)

Este notebook consome o CSV **paciente-dia** (export Supabase ou arquivo de exemplo em `ml/data/sample_glicnutri_patient_day.csv`), executa validação básica, EDA resumida e quatro tarefas:

1. **Classificação** — dia com glicemia média elevada (rótulo: `glucose_mean_mg_dl >= 150`).
2. **Regressão** — prever `glucose_mean_mg_dl` a partir de hábitos/resumos diários.
3. **Clusterização** — grupos de dias por padrão nutricional/leituras.
4. **Similaridade** — vizinhos mais próximos no espaço de features (NearestNeighbors).

Ao final, os artefatos são gravados em `backend/artifacts/` para uso pela API FastAPI (`POST /predict`). Para dados reais, gere o CSV com `ml/scripts/export_supabase_csv.py` e aponte `CSV_PATH` abaixo.

In [ ]:
from pathlib import Path
import json
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "backend" / "artifacts").exists():
    ROOT = ROOT.parent
if not (ROOT / "backend" / "artifacts").exists():
    ROOT = Path.cwd()

CSV_PATH = ROOT / "ml" / "data" / "sample_glicnutri_patient_day.csv"
ARTIFACTS = ROOT / "backend" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    "n_leituras_glicemia",
    "carbs_sum_g",
    "kcal_sum",
    "protein_sum_g",
    "fat_sum_g",
    "n_refeicoes_ia",
    "n_eventos_medicacao",
]

print("ROOT:", ROOT)
print("CSV:", CSV_PATH.resolve())

In [ ]:
df = pd.read_csv(CSV_PATH)
df["dia"] = pd.to_datetime(df["dia"])
assert not df[FEATURE_COLS].isna().all().all(), "Dataset sem features numéricas"
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)
df["target_glucose_high"] = (df["glucose_mean_mg_dl"] >= 150).astype(int)
df.describe()

## EDA (exploração rápida)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["glucose_mean_mg_dl"].hist(ax=axes[0], bins=12, color="#159365", edgecolor="white")
axes[0].set_title("Distribuição glicemia média (mg/dL)")
axes[1].scatter(df["carbs_sum_g"], df["glucose_mean_mg_dl"], alpha=0.7, c="#159365")
axes[1].set_xlabel("Carboidratos (g/dia)")
axes[1].set_ylabel("Glicemia média")
plt.tight_layout()
plt.show()

corr = df[["glucose_mean_mg_dl"] + FEATURE_COLS].corr()
corr["glucose_mean_mg_dl"].sort_values(ascending=False)

## Tarefa 1 — Classificação

In [ ]:
X = df[FEATURE_COLS]
y_cls = df["target_glucose_high"]
X_train, X_test, y_train, y_test = train_test_split(X, y_cls, test_size=0.25, random_state=42, stratify=y_cls if y_cls.nunique() > 1 else None)

clf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])
clf_pipe.fit(X_train, y_train)
pred = clf_pipe.predict(X_test)
print("Acurácia:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

## Tarefa 2 — Regressão

In [ ]:
y_reg = df["glucose_mean_mg_dl"]
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.25, random_state=42)

reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])
reg_pipe.fit(X_train, y_train)
pred_r = reg_pipe.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred_r))
print("R²:", r2_score(y_test, pred_r))

## Tarefa 3 — Clusterização

In [ ]:
k = min(3, len(df))
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X)
df["cluster"] = km.labels_
df["cluster"].value_counts()

## Tarefa 4 — Similaridade / recomendação por vizinhança

In [ ]:
nn = NearestNeighbors(n_neighbors=min(3, len(X)))
nn.fit(X)
dist, idx = nn.kneighbors(X.iloc[[0]], return_distance=True)
print("Distâncias:", dist)
print("Índices vizinhos:", idx)

## Persistência (joblib) — alinhado à API `POST /predict`

In [ ]:
joblib.dump(clf_pipe, ARTIFACTS / "model_classification.joblib")
joblib.dump(reg_pipe, ARTIFACTS / "model_regression.joblib")
joblib.dump(km, ARTIFACTS / "model_clustering.joblib")
joblib.dump(nn, ARTIFACTS / "model_similarity.joblib")
joblib.dump(X.values, ARTIFACTS / "similarity_train_matrix.joblib")

meta = {
    "feature_columns": FEATURE_COLS,
    "target_classification": "target_glucose_high (glucose_mean_mg_dl >= 150)",
    "target_regression": "glucose_mean_mg_dl",
    "csv_source": str(CSV_PATH.resolve()),
}
with open(ARTIFACTS / "training_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Artefatos salvos em", ARTIFACTS)